In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "..."
os.environ["OPENAI_API_BASE"] = "..."

import openai
from langchain.llms import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

import json

import chromadb

In [2]:
# Defined parameters used in the project

model_name = "text-embedding-ada-002"

embedding_function = OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name = model_name
)

collection_name = "homematch"

#### Generating a listing of real estate and saving to a JSON file

In [ ]:
def generate_listing():
    try:
        prompt = """You are a professional real estate copywriter.
Below are an example of a well-structured real estate listing in JSON format:

---
{
  "Neighborhood": "Green Oaks",
  "Price": "$800,000",
  "Bedrooms": 3,
  "Bathrooms": 2,
  "House Size": "2,000 sqft",
  "Description": "Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.",
  "Neighborhood Description": "Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze."
}
---

Now, create a NEW and ORIGINAL real estate listing following the same JSON format.

Vary the **Price**, **Bedrooms**, **Bathrooms**, and **House Size** in a realistic way. Examples:
- Price can range from $150,000 (small apartments) up to $2,500,000 (luxury homes).
- Bedrooms can range from 1 to 6.
- Bathrooms can range from 1 to 5.
- House Size can range from 600 sqft (small condo) up to 5,000 sqft (large estate).

Ensure that the listing feels realistic, friendly, and slightly persuasive. Mention appealing features appropriate to the price and size (e.g., luxury features for high-end homes, cozy features for smaller ones).

**Bedrooms** and **Bathrooms** must be integer values. 
Output ONLY the JSON object without any extra commentary.
"""

        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=800
        )
        listing_text = response['choices'][0]['message']['content']
        
        # Try parsing the output as JSON
        listing_json = json.loads(listing_text)
        return listing_json
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def generate_multiple_listings(n=10):
    listings = []
    for i in range(n):
        listing = generate_listing()
        if listing:
            listings.append(listing)
            print(f"Generated listing {i+1}: {listing}\n{'='*50}\n")
    return listings

In [ ]:
listings_json = generate_multiple_listings(20)

In [ ]:
if os.path.isdir('data') == False:
    os.mkdir('data')

with open('data/real_estates_listings.json', 'w') as json_file:
        json.dump(listings_json, json_file)
        print(f"Listings saved as JSON file")

#### ChromaBD inputs and simple querying

In [ ]:
# You may need to upload a generated listing, if you starts with this step

with open('.data/real_estates_listings.json', 'r') as f:
    listings = json.load(f)
    print(f"Data uploaded")

In [ ]:
client = chromadb.PersistentClient(path="./data")

try:
    collection = client.get_collection(name=collection_name, embedding_function=embedding_function)
except Exception as e:
    collection = client.create_collection(name=collection_name, embedding_function=embedding_function)

In [ ]:
def get_embeddings(texts):
    embeddings = []
    for text in texts:
        response = openai.Embedding.create(
            input=text,
            model = model_name
        )
        embeddings.append(response['data'][0]['embedding'])
    return embeddings

In [ ]:
ids = []
metadatas = []
documents = []
neighborhood = []
price = []
bedrooms = []
bathrooms = []
house_size = []
description = []
n_description = []

for idx, listing in enumerate(listings_json):
    # Create a full text to embed
    document_text = (
        f"Neighborhood: {listing['Neighborhood']}\n"
        f"Price: {listing['Price']}\n"
        f"Bedrooms: {listing['Bedrooms']}\n"
        f"Bathrooms: {listing['Bathrooms']}\n"
        f"House Size: {listing['House Size']}\n"
        f"Description: {listing['Description']}\n"
        f"Neighborhood Description: {listing['Neighborhood Description']}"
    )

    ids.append(f"listing-{idx}")
    metadatas.append(listing)
    documents.append(document_text)
    neighborhood.append(listing['Neighborhood'])
    price.append(float(listing['Price'].replace('$','').replace(',','')))
    bedrooms.append(int(listing['Bedrooms']))
    bathrooms.append(int(listing['Bathrooms']))
    house_size.append(float(listing['House Size'].replace(' sqft','').replace(',','')))
    description.append(listing['Description'])
    n_description.append(listing['Neighborhood Description'])


embeddings = get_embeddings(documents)

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=[
        {
            "neighborhood": neighborhood[i],
            "price_usd": price[i],
            "bedrooms": bedrooms[i],
            "bathrooms": bathrooms[i],
            "house_size_sqft": house_size[i],
            "description": description[i],
            "neighborhood_description": n_description[i]
        }
        for i in range(len(ids))
    ]
)

print("Listings successfully embedded and stored in ChromaDB!")
    

In [ ]:
# Simple checking

print('No of listings: ' + str(collection.count()))
collection.peek(1)

In [ ]:
# Simple querying

query_text = "presitgious and exclusive"
query_embedding = get_embeddings([query_text])[0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,  # top 3 similar homes
    include=["documents", "metadatas", "distances"]
)

print(results['ids'])
print(results['distances'])

for i in results['documents'][0]:
    print(i)
    print('---------')

for i in results['metadatas'][0]:
    print(i)
    print('---------')

#### Advanced searching

In [3]:
# Access to database

client = chromadb.PersistentClient(path="./data")
collection = client.get_collection(name=collection_name, embedding_function=embedding_function)

In [4]:
def parse_query_into_features(query_text):
    prompt = f"""
You are a smart real estate assistant. 
Analyze the following user query carefully and extract structured filters.

Important Instructions:
- If the user says "about X", calculate 80% of X as minimum, 120% of X as maximum.
- Only return numeric values (integers) without units (e.g., no "sqft" or "$").
- If any value is missing or not specified, output null.
- Output must strictly be a valid JSON object without any code blocks or markdown formatting.

Extract the following fields:
- min_price_usd (integer or null)
- max_price_usd (integer or null)
- bedrooms (integer or null)
- bathrooms (integer or null)
- min_house_size (integer or null)
- max_house_size (integer or null)

Example format:
{{
  "min_price_usd": 1000000,
  "max_price_usd": 2000000,
  "bedrooms": 3,
  "bathrooms": 2,
  "min_house_size": 2000,
  "max_house_size": 2500
}}

Examples:
[Query: "Luxury estate with pool in Sunset Hills to 2 million dollars and not less than 1,500 sqft, 3 bedrooms and 2 bathrooms"]
[Result example:
{{
  "min_price_usd": None,
  "max_price_usd": 2000000,
  "bedrooms": 3,
  "bathrooms": 2,
  "min_house_size": 1500,
  "max_house_size": None
}}] 

[Query: "Modern house between 1.3 and 1.8 million, 3 bedrooms, max 3,000 sqft"]
[Result example:
{{
  "min_price_usd": 1300000,
  "max_price_usd": 1800000,
  "bedrooms": 3,
  "bathrooms": None,
  "min_house_size": None,
  "max_house_size": 3000
}}]
End of examples

Query: "{query_text}"
"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You extract structured real estate search filters."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            max_tokens=300
        )

        content = response['choices'][0]['message']['content']

        features = json.loads(content)

    except Exception as e:
        print(f"Error parsing model response: {e}")
        print("Raw content:", content)  # for debug
        features = {}
    
    return features

In [5]:
def personalized_query_results(query_text, tolerance=True, top_k=3):
    
    # Tolerance for number of bedrooms and bathrooms
    if tolerance ==True:
        tol = 1
    else:
        tol = 0

    parsed_features = parse_query_into_features(query_text)
    and_filters = []

    # Price
    if parsed_features.get('min_price_usd') is not None:
        and_filters.append({
            "price_usd": {"$gte": parsed_features['min_price_usd']}
        })
    if parsed_features.get('max_price_usd') is not None:
        and_filters.append({
            "price_usd": {"$lte": parsed_features['max_price_usd']}
        })

    # Bedrooms
    if parsed_features.get('bedrooms') is not None:
        if tolerance:
            and_filters.append({
                "bedrooms": {
                    "$gte": parsed_features['bedrooms'] - tol
                }
            })
            and_filters.append({
                "bedrooms": {
                    "$lte": parsed_features['bedrooms'] + tol
                }
            })
        else:
            and_filters.append({
                "bedrooms": {
                    "$eq": parsed_features['bedrooms']
                }
            })

    # Bathrooms
    if parsed_features.get('bathrooms') is not None:
        if tolerance:
            and_filters.append({
                "bathrooms": {
                    "$gte": parsed_features['bathrooms'] - tol
                }
            })
            and_filters.append({
                "bathrooms": {
                    "$lte": parsed_features['bathrooms'] + tol
                }
            })
        else:
            and_filters.append({
                "bathrooms": {
                    "$eq": parsed_features['bathrooms']
                }
            })

    # House size
    if parsed_features.get('min_house_size') is not None:
        and_filters.append({
            "house_size_sqft": {"$gte": parsed_features['min_house_size']}
        })
    if parsed_features.get('max_house_size') is not None:
        and_filters.append({
            "house_size_sqft": {"$lte": parsed_features['max_house_size']}
        })

    # Final filter
    chroma_filter = {"$and": and_filters} if and_filters else {}

    # Query
    results = collection.query(
        query_texts=[query_text],
        n_results=top_k,
        where=chroma_filter,
        include=["documents", "metadatas"]
    )

    return results

In [6]:
def summarize_listing(description, neighborhood_description, query_text):
    prompt = f"""
Summarize the following real estate listing in 1-2 sentences. Highlight key features such as luxury, space, views, neighborhood benefits, or any unique selling points, but stress importance of elements required in {query_text}. Keep the tone natural, informative, and engaging.

Listing description:
{description}

Neighborhood description:
{neighborhood_description}

Summary:"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            max_tokens=200
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("Error generating summary:", e)
        return "A beautiful home in a desirable neighborhood."

In [7]:
def display_results(results, query_text):

    listings = results.get("metadatas", [[]])[0]

    if not listings:
        print("\n No matching listings found. Try adjusting your query or reducing filters.\n")
        return

    print("\n Top Matching Listings:\n" + "-" * 40)

    for i, (metadata, doc) in enumerate(zip(results['metadatas'][0], results['documents'][0]), 1):
        print(f"Listing {i}:")
        print(f"Neighborhood: {metadata.get('neighborhood')}")
        print(f"Price: ${metadata.get('price_usd'):,.0f}")
        print(f"Bedrooms: {metadata.get('bedrooms')}")
        print(f"Bathrooms: {metadata.get('bathrooms')}")
        print(f"House Size: {metadata.get('house_size_sqft')} sqft")
        description = metadata.get('description')
        neighborhood_description = metadata.get('neighborhood_description')
        summary = summarize_listing(description, neighborhood_description, query_text)
        print(f"Summary: {summary}")
        print("-" * 40)

In [8]:
query_text = "Luxury estate with pool in Sunset Hills to 2 million dollars and not less than 1,500 sqft"
results = personalized_query_results(query_text, tolerance=True) #Tolerance == True increases searching with widened number of bedrooms and bathrooms
display_results(results, query_text)


 Top Matching Listings:
----------------------------------------
Listing 1:
Neighborhood: Sunset Hills
Price: $1,500,000
Bedrooms: 4
Bathrooms: 4
House Size: 3500.0 sqft
Summary: Luxury living awaits in this grand 4-bedroom, 4-bathroom estate in Sunset Hills, featuring high-end finishes, a gourmet chef's kitchen, and a private backyard oasis with a sparkling pool and spa. Embrace the opulence of this prestigious neighborhood with exclusive country clubs, fine dining restaurants, top-rated schools, and panoramic city views, offering the epitome of sophisticated living.
----------------------------------------
Listing 2:
Neighborhood: Sunset Hills
Price: $1,500,000
Bedrooms: 5
Bathrooms: 4
House Size: 4500.0 sqft
Summary: Luxury estate in Sunset Hills offers a spacious 5-bedroom, 4-bathroom home with high-end finishes, gourmet kitchen, and a private backyard oasis with pool and spa. Located in a prestigious neighborhood with stunning city skyline views, fine dining, and boutique shoppin

In [9]:
query_text = "Modern house with 2 bedrooms for young man, active in business and looking for easy access to entertainment"
results = personalized_query_results(query_text, tolerance=True) #Tolerance == True increases searching with widened number of bedrooms and bathrooms
display_results(results, query_text)


 Top Matching Listings:
----------------------------------------
Listing 1:
Neighborhood: Seaside Cove
Price: $450,000
Bedrooms: 2
Bathrooms: 2
House Size: 1200.0 sqft
Summary: This 2-bedroom beachside retreat in Seaside Cove offers stunning ocean views, a private backyard oasis with a hot tub, and easy access to sandy beaches, seaside cafes, and boutique shops. Perfect for a young businessman looking for a modern home with entertainment options in a vibrant coastal community.
----------------------------------------
Listing 2:
Neighborhood: Seaside Cove
Price: $450,000
Bedrooms: 2
Bathrooms: 2
House Size: 1200.0 sqft
Summary: This 2-bedroom home in Seaside Cove offers a cozy atmosphere with stunning ocean views, a fireplace, updated kitchen, and a private backyard oasis with a deck for entertaining. Located in a charming coastal community with sandy beaches, seaside cafes, and boutique shops, it is perfect for a young businessman looking for easy access to entertainment and outdoor a

In [10]:
query_text = "Modern house in prestigous place. No limitiation about price. Seaside view. Home for family with 2 childrens"
results = personalized_query_results(query_text, tolerance=True) #Tolerance == True increases searching with widened number of bedrooms and bathrooms
display_results(results, query_text)


 Top Matching Listings:
----------------------------------------
Listing 1:
Neighborhood: Seaside Estates
Price: $1,500,000
Bedrooms: 5
Bathrooms: 4
House Size: 4500.0 sqft
Summary: Experience luxury living in this stunning 5-bedroom, 4-bathroom estate in the prestigious Seaside Estates, boasting a gourmet chef's kitchen, private home theater, and a spacious master suite with ocean views. Enjoy outdoor living with a pool, BBQ, and lush landscaping, all in a sought-after neighborhood with private beach access, upscale amenities, and top-rated schools.
----------------------------------------
Listing 2:
Neighborhood: Seaside Retreat
Price: $550,000
Bedrooms: 4
Bathrooms: 3
House Size: 2500.0 sqft
Summary: Experience luxury coastal living in this prestigious Seaside Retreat home, featuring 4 bedrooms, 3 bathrooms, a gourmet kitchen, and a resort-style backyard with a pool and outdoor kitchen. Enjoy breathtaking ocean views, private beaches, upscale dining options, and exclusive amenities